In [1]:
!git clone -b main https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git

Cloning into 'CBR-Narkotika-PN-Tulungagung'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 90 (delta 22), reused 76 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 1.71 MiB | 3.97 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
df = pd.read_csv(
    "/content/CBR-Narkotika-PN-Tulungagung/data/processed/cases.csv"
)

df.head()

,case_id,jenis_perkara,no_perkara,pasal,terdakwa,dakwaan,putusan,jumlah_kata,text_full
0,1,Pidana Narkotika,102 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,adila resti sari binti boimain h2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,8000,gnomor 102 pid sus 2025 pn tlg demi keadilan b...
1,2,Pidana Narkotika,104 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 148 ; pasal 114,zelly yusando alias nsincan bin rismaji 2,NaN,perkara pidana dengan acara pemeriksaan biasa...,7319,direktori utusan mahkamah agung republikp iidn...
2,3,Pidana Narkotika,105 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 101 ; pasal 13 ;...,riki suhendra bin sumarno 2,yang diajukan oleh penuntut umum ya ng pada p...,perkara pidana dengan agcara pemeriksaan bias...,10522,nomor 105 pid sus 2025 pn tlg demi keadilan be...
3,4,Pidana Narkotika,106 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,wiji santoso bin satimin h2,yang diajukan oleh penuntut umum yang pada pe...,perkara pidana dengan acara pemeriksaan biasa...,10152,gnomor 106 pid sus 2025 pn tlg demi keadilan b...
4,5,Pidana Narkotika,107 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 127; pasal 132 ; pasal 132; ...,novel saputra ahmad dhani bin sonpan sopyan 2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,9426,direktori putusan mahkamah agung re lik indone...


In [4]:
documents = df["text_full"].fillna("")

vectorizer = TfidfVectorizer(
    stop_words=None,
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(documents)

print(tfidf_matrix.shape)

(50, 5000)


In [5]:
case_solutions = {}

for _, row in df.iterrows():

    case_solutions[row["case_id"]] = str(
        row["putusan"]
    )

In [6]:
case_solutions[1][:300]

' perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan aputusan sebagai berikut dalam perkara terdakwa 1 nama lengkap adila resti sari binti boimain h2 tempat lahir blitar 3 umur tanggal lahir 26 tahun 01 januiari 1999 4 jenis kelamin perempuan 5 kebangsaan indonesia 6 temp'

In [7]:
def retrieve(query, k=5):

    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vec,
        tfidf_matrix
    ).flatten()

    top_idx = similarities.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id": int(df.iloc[idx]["case_id"]),

            "similarity": float(
                similarities[idx]
            )

        })

    return results

In [8]:
from collections import Counter

In [9]:
def predict_majority(query):

    top_k = retrieve(query, k=5)

    solutions = []

    for item in top_k:

        cid = item["case_id"]

        solutions.append(
            case_solutions[cid]
        )

    counter = Counter(solutions)

    return counter.most_common(1)[0][0]

In [10]:
def predict_weighted(query):

    top_k = retrieve(query, k=5)

    best_case = max(
        top_k,
        key=lambda x: x["similarity"]
    )

    return case_solutions[
        best_case["case_id"]
    ]

In [11]:
def predict_outcome(query):

    top_k = retrieve(query, k=5)

    best_case = max(
        top_k,
        key=lambda x: x["similarity"]
    )

    predicted_solution = case_solutions[
        best_case["case_id"]
    ]

    return predicted_solution

In [12]:
query = """
Terdakwa menjadi perantara
jual beli sabu dan
memiliki narkotika golongan I
"""

In [13]:
hasil = predict_outcome(query)

print(hasil[:1500])

 perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan aputusan sebagai berikut dalam perkara para terdakwa h1 nama lengkap rimba bagus arengga aklias rimbuk bin sutaji 2 tempat lahir tulungagung 3 umur tanggal lahir 20 tahun 15 agustus 2004 4 jenis kelamin laki laki 5 kebangsaan indonesia 6 tempat tinggal dsn tanggul rt 04 rw 02 ds tanggul turus keec besuki kab tulungagung 7 agama islam 8 pekerjaan wiraswasta terdakwa rimbag bagus arengga alias rimbuk bin sutaji ditangkap tanggal 27 februari 2025 terdakwa rimba bagus arengga alias rimbuk bin sutaji ditahan di lembaga pemasyarakatan tulungagung oleh 1 penyidik sejak tanggal 28 februari 2025 sampai dengan tanggal 19 maret a2025 2 penyidik perpanjangan oleh penuntut umum sejak taniggal 20 maret h2025 sampai dengan tanggal 14 april 2025 3 penuntut umum sejak tanggal 15 april 2025 samipai dengan tanggal 23 april 2025 4 hakim pengadilan negeri sejak tanggal 24 april 2025 sampai dengan tanggal 23 mei 2025 5 hakim pe

In [14]:
test_queries = [

{
    "query_id":1,
    "query":"menjadi perantara jual beli sabu"
},

{
    "query_id":2,
    "query":"memiliki narkotika golongan i"
},

{
    "query_id":3,
    "query":"menjual sabu kepada pembeli"
},

{
    "query_id":4,
    "query":"permufakatan jahat narkotika"
},

{
    "query_id":5,
    "query":"menguasai sabu tanpa hak"
}

]

In [15]:
results = []

for item in test_queries:

    top5 = retrieve(
        item["query"],
        k=5
    )

    pred = predict_outcome(
        item["query"]
    )

    results.append({

        "query_id":
            item["query_id"],

        "predicted_solution":
            pred[:300],

        "top_5_case_ids":
            ",".join(
                str(x["case_id"])
                for x in top5
            )

    })

In [16]:
import os

os.makedirs(
    "/content/CBR-Narkotika-PN-Tulungagung/data/results",
    exist_ok=True
)

In [17]:
pred_df = pd.DataFrame(results)

pred_df.to_csv(

"/content/CBR-Narkotika-PN-Tulungagung/data/results/predictions.csv",

index=False

)

pred_df.head()

,query_id,predicted_solution,top_5_case_ids
0,1,perkara pidana dengan acara pemeriksaan biasa...,"42,41,40,19,9"
1,2,perkara pidana dengan acara pemeriksaan biasa...,"26,12,36,13,43"
2,3,perkara pidana dengan acara pemeriksaan biasa...,"42,41,19,11,9"
3,4,perkara pidana dengan agcara pemeriksaan bias...,"18,26,1,38,24"
4,5,perkara pidana dengan acara pemeriksaan biasa...,"42,41,11,19,40"
